# 🧭 Autogen 멀티 에이전트 환불 처리: Travel × FlightsRefunder × User(Handoff)

이 노트북은 **2개의 에이전트**가 역할을 분담해
**항공권 환불**을 협업으로 처리하고,
필요 시 **사용자에게 handoff**로 정보를 수집하며
완료 시 **"TERMINATE"** 또는 **사용자 handoff 도달**로 종료하는 예제입니다.

---

## 🧩 구성 요소

| 구성 요소 | 역할 |
|---|---|
| **travel_agent** | 환불 요청 접수, 정보 수집(예약번호 등), 흐름 총괄 |
| **flights_refunder** | 환불 전담 에이전트, `refund_flight` **툴 호출** |
| **refund_flight(tool)** | 입력된 항공권 ID 환불(모의 함수) |
| **Swarm** | 분산 협업(에이전트가 **handoff**로 서로에게 일을 넘김) |
| **HandoffMessage** | 사용자 ↔ 에이전트 간, 에이전트 ↔ 에이전트 간 **바통(메시지)** |
| **HandoffTermination("user")** | 메시지 대상이 **user**인 handoff 발생 시 종료 |
| **TextMentionTermination("TERMINATE")** | 응답에 **"TERMINATE"** 등장 시 종료 |
| **Console** | 스트리밍 메시지를 보기 좋게 출력 |

---

## ⚙️ 동작 흐름

1. **Travel**이 환불 진행 안내 → **예약번호 요청** → 필요 시 **handoff → user**  
2. 사용자가 예약번호 입력 → **handoff → 해당 에이전트**(요청자)  
3. **FlightsRefunder**가 `refund_flight` **툴 호출** → 환불 처리 → **handoff → travel_agent**(최종 안내)  
4. **Travel**이 결과 통지 및 마무리 → **"TERMINATE"** 출력 → 종료  
   - 또는 설계에 따라 **사용자 handoff 도달 시 종료**

---

## Flight refund process

In [ ]:

from typing import Any, Dict, List

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import HandoffTermination, TextMentionTermination
from autogen_agentchat.messages import HandoffMessage
from autogen_agentchat.teams import Swarm
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient
from dotenv import load_dotenv
import os

load_dotenv()

api_version = os.getenv("AZURE_OPENAI_API_VERSION")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
azure_openai_chat_completion_client = AzureOpenAIChatCompletionClient(
            model=deployment_name,
            azure_endpoint=azure_endpoint,
            api_version=api_version,
            api_key=api_key,
        )

def refund_flight(flight_id: str) -> str:
    """Refund a flight"""
    return f"Flight {flight_id} refunded"

In [ ]:
model_client = azure_openai_chat_completion_client

travel_agent = AssistantAgent(
    "travel_agent",
    model_client=model_client,
    handoffs=["flights_refunder", "user"],
    system_message="""You are a travel agent.
    The flights_refunder is in charge of refunding flights.
    If you need information from the user, you must first send your message, then you can handoff to the user.
    Use TERMINATE when the travel planning is complete.""",
)

flights_refunder = AssistantAgent(
    "flights_refunder",
    model_client=model_client,
    handoffs=["travel_agent", "user"],
    tools=[refund_flight],
    system_message="""You are an agent specialized in refunding flights.
    You only need flight reference numbers to refund a flight.
    You have the ability to refund a flight using the refund_flight tool.
    If you need information from the user, you must first send your message, then you can handoff to the user.
    When the transaction is complete, handoff to the travel agent to finalize.""",
)

termination = HandoffTermination(target="user") | TextMentionTermination("TERMINATE")
team = Swarm([travel_agent, flights_refunder], termination_condition=termination)


In [ ]:
task = "항공권을 환불해야 합니다."


async def run_team_stream() -> None:
    task_result = await Console(team.run_stream(task=task))
    last_message = task_result.messages[-1]

    while isinstance(last_message, HandoffMessage) and last_message.target == "user":
        user_message = input("User: ")

        task_result = await Console(
            team.run_stream(task=HandoffMessage(source="user", target=last_message.source, content=user_message))
        )
        last_message = task_result.messages[-1]


await run_team_stream()

# 📊 Autogen 멀티 에이전트 리서치: Planner × Financial × News × Writer (Swarm)

이 노트북은 **4개의 에이전트**가 역할을 분담해
**TSLA(테슬라) 주식 시장 조사**를 협업으로 수행하고,
최종적으로 **"TERMINATE"** 신호로 종료되는 예제입니다.

---

## 🧩 구성 요소

| 구성 요소 | 역할 |
|---|---|
| **planner** | 전체 연구 계획 수립/조율, 단계별 **handoff** 지시 |
| **financial_analyst** | `get_stock_data` **툴 호출**로 재무 지표 분석 |
| **news_analyst** | `get_news` **툴 호출**로 최신 뉴스 수집·요약 |
| **writer** | 분석 결과를 **최종 리포트** 형태로 정리 |
| **Swarm** | 에이전트 간 **handoff 기반 분산 협업** 오케스트레이션 |
| **TextMentionTermination("TERMINATE")** | 메시지에 **"TERMINATE"** 등장 시 종료 |
| **Console** | 스트리밍 대화 로그를 보기 좋게 출력 |
| **get_stock_data(tool)** | 주가/거래량/PER/시가총액 (모의 데이터) |
| **get_news(tool)** | 회사 관련 최신 뉴스 목록 (모의 데이터) |

---

## ⚙️ 동작 흐름

1. **Planner**가 연구 **계획**을 제시하고, 이번 단계의 담당 에이전트를 **단일 선택**하여 **handoff**  
2. **Financial Analyst**가 `get_stock_data`로 **재무 지표 분석** → **planner로 handoff**  
3. **Planner**가 다음 단계로 **News Analyst**에게 handoff  
4. **News Analyst**가 `get_news`로 **뉴스 수집/요약** → **planner로 handoff**  
5. **Planner**가 **Writer**에게 handoff  
6. **Writer**가 내용을 **완결된 리포트**로 정리 → **planner로 handoff**  
7. **Planner**가 결과를 확인/종결하고 **"TERMINATE"** 출력 → 종료

---

## Stock Research

In [ ]:
async def get_stock_data(symbol: str) -> Dict[str, Any]:
    """Get stock market data for a given symbol"""
    return {"price": 180.25, "volume": 1000000, "pe_ratio": 65.4, "market_cap": "700B"}


async def get_news(query: str) -> List[Dict[str, str]]:
    """Get recent news articles about a company"""
    return [
        {
            "title": "테슬라, 사이버트럭 생산 확대",
            "date": "2024-03-20",
            "summary": "Tesla는 강력한 수요를 충족하기 위해 기가팩토리 텍사스의 사이버트럭 제조 역량을 강화합니다.",
        },
        {
            "title": "가능성을 보여주는 테슬라 FSD 베타 버전",
            "date": "2024-03-19",
            "summary": "최신 완전 자율주행 베타 버전에서는 도심 내비게이션 및 안전 기능이 크게 개선되었습니다.",
        },
        {
            "title": "Model Y, 글로벌 전기차 판매 1위",
            "date": "2024-03-18",
            "summary": "Tesla의 Model Y가 전 세계에서 가장 많이 팔린 전기자동차가 되어 상당한 시장 점유율을 차지했습니다.",
        },
    ]

In [ ]:
model_client = azure_openai_chat_completion_client
planner = AssistantAgent(
    "planner",
    model_client=model_client,
    handoffs=["financial_analyst", "news_analyst", "writer"],
    system_message="""You are a research planning coordinator.
    Coordinate market research by delegating to specialized agents:
    - Financial Analyst: For stock data analysis
    - News Analyst: For news gathering and analysis
    - Writer: For compiling final report
    Always send your plan first, then handoff to appropriate agent.
    Always handoff to a single agent at a time.
    Use TERMINATE when research is complete.""",
)

financial_analyst = AssistantAgent(
    "financial_analyst",
    model_client=model_client,
    handoffs=["planner"],
    tools=[get_stock_data],
    system_message="""You are a financial analyst.
    Analyze stock market data using the get_stock_data tool.
    Provide insights on financial metrics.
    Always handoff back to planner when analysis is complete.""",
)

news_analyst = AssistantAgent(
    "news_analyst",
    model_client=model_client,
    handoffs=["planner"],
    tools=[get_news],
    system_message="""You are a news analyst.
    Gather and analyze relevant news using the get_news tool.
    Summarize key market insights from news.
    Always handoff back to planner when analysis is complete.""",
)

writer = AssistantAgent(
    "writer",
    model_client=model_client,
    handoffs=["planner"],
    system_message="""You are a financial report writer.
    Compile research findings into clear, concise reports.
    Always handoff back to planner when writing is complete.""",
)

In [ ]:
# Define termination condition
text_termination = TextMentionTermination("TERMINATE")
termination = text_termination

research_team = Swarm(
    participants=[planner, financial_analyst, news_analyst, writer], termination_condition=termination
)

task = "TSLA 주식에 대한 시장 조사 수행"
response = await Console(research_team.run_stream(task=task))

In [ ]:
print(response.messages[-7].content)